- 110e3 DN in 60 seconds
- This star is 149180247107173789 in PS DR2. It has r = 13.279 in APASS DR10.
- Gain = 2.48 e/DN
- ZP = 1.24e9 

- Background noise is 1.8 * 2.48 e/pixel

- 1 arcsec = 13.3 pixel
- For 1 arcsec seeing number of pixels in 2 FWHM diameter circle is

In [ ]:
from math import *

def dex(x):
    return 10 ** x
def square(x):
    return x * x

gain = 2.48
print("gain = %.1f electron/DN" % gain)

ZP = 110e3 * gain / 60 * (10 ** (0.4 * 13.279))
print("ZP = %.2e electron/s" % ZP)

skymagnitude = 20
print("sky magnitude = %.1f mag/arcsec²")

readnoise = 2.6
print("read noise = %.1f electron" % readnoise)

exposuretime = 60
print("exposuretime = %.1f s" % exposuretime)

pixelscale = 0.075

backgroundperpixel = ZP * dex(-0.4 * skymagnitude) * square(pixelscale) * exposuretime
print("background per pixel = %.1f electron" % backgroundperpixel)

pixelnoise = sqrt(square(readnoise) + backgroundperpixel)
print("pixel noise =  = %.1f electron = %.1f DN" % (pixelnoise, pixelnoise / gain))

seeingfwhm = 1.0
print("seeing = %.1f arcsec" % seeingfwhm)
seeingsigma = seeingfwhm / (2 * sqrt(log(2)))

apertureradius = 0.5
print("aperture radius = %.2f arcsec = %.2f seeing" % (apertureradius, apertureradius / seeingfwhm))
aperturecorrection = 1 - exp(- (apertureradius / seeingsigma) ** 2)
print("aperture correction = %.2f" % aperturecorrection)


#pixelnoise = 1.8 * gain

pixelsperaperture = pi * square(apertureradius / pixelscale)
aperturenoise = sqrt(pixelsperaperture) * pixelnoise
print("aperture noise = %.1f electron" % aperturenoise)

equivalentmagnitude = -2.5 * log10(square(aperturenoise) / (ZP * exposuretime * aperturecorrection))
print("magnitude equivalent to aperture noise = %.2f" % equivalentmagnitude)


import matplotlib.pyplot as plt
import numpy as np

def totalsignal(m):
    return ZP * (10 ** (-0.4 * m)) * exposuretime

def snr(m):
    signal = aperturecorrection * totalsignal(m)
    noise = np.sqrt(aperturenoise ** 2 + signal)
    return signal / noise

m = np.linspace(10, 22, 100)

plt.plot(m, snr(m), label="%.0f s" % exposuretime)
plt.plot(m, snr(m) * sqrt(10), label=r"$10 \times %.0f$ s" % exposuretime)
plt.plot(m, snr(m) * sqrt(100), label=r"$100 \times %.0f$ s" % exposuretime)
plt.legend()
plt.yscale("log")
plt.grid(which="both", axis="both")
plt.xlim(12, 22)
plt.xticks(np.linspace(12, 22, 11))
plt.xlabel(r"$r$")
plt.ylabel(r"SNR in $I$")
plt.title(r"TEQUILA on COLIBRÍ")
plt.savefig("snr.png")
plt.show()

meanDN = totalsignal(m) / pixelsperaperture / gain

plt.plot(m, meanDN)
plt.yscale("log")
plt.grid(which="both", axis="both")
plt.xlim(12, 22)
plt.xlabel(r"$r$")
plt.ylabel(r"Mean DN in 60 seconds")
plt.title(r"TEQUILA on COLIBRÍ")
plt.show()
